# Rabbit Hole Finder: FULL build on Kaggle (2× T4)

Builds the FULL dataset with the same Dagster pipeline as `make pipeline`, minus the final
Postgres load (that happens on your machine). Output: `/kaggle/working/rabbithole_full/`.

**Settings before running:** Accelerator **GPU T4 ×2**, Internet **on** (needs a phone-verified
account). Then **Save Version → Save & Run All (Commit)** so it runs in the background (~2–4 h).

What runs where:
- clickstream download + parse, FULL subset, link and label edges: CPU
- article text: streamed from the Hugging Face `wikimedia/wikipedia` dump (no big single download)
- article + chunk embeddings: **both GPUs**, fp16, one encoder process per GPU
- Leiden sweep, rabbit-holeness, pruned graph, UMAP map: CPU

In [ ]:
# Settings: the only cell you might edit.
FULL_MIN_VIEWS = 1000        # >=1000 views in the feature month ~ 654k articles (spec default)
CLICKSTREAM_MONTHS = "2026-07,2026-08,2026-09"   # same months as the local DEV build
MAX_CHUNKS_PER_ARTICLE = 8
LEIDEN_RESOLUTIONS = [1, 2, 3, 4, 6, 8, 12, 20, 30, 50, 80]  # FULL needs finer settings than DEV
REPO = "https://github.com/Siddhartha121220/RabbitHole_Finder.git"
BRANCH = "main"

In [ ]:
!nvidia-smi -L
!df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null
!python --version

In [ ]:
import os, subprocess, sys

def run(*cmd, **kw):
    return subprocess.run(list(map(str, cmd)), check=True, **kw)

run("git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/kaggle/working/src")

# The project needs Python >= 3.12 (scipy 1.18); Kaggle's own Python may be older, so build the
# exact locked environment with uv on 3.12 (uv fetches it if needed). The venv lives in
# /kaggle/temp so it does not count against the 20 GB output limit.
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
VENV = f"{TMP}/venv"
PY = f"{VENV}/bin/python"
os.environ.update(UV_PROJECT_ENVIRONMENT=VENV, UV_CACHE_DIR=f"{TMP}/uv-cache")
run(sys.executable, "-m", "pip", "install", "-q", "uv")
run("uv", "sync", "--locked", "--no-dev", "--no-install-package", "torch", "--python", "3.12",
    "--project", "/kaggle/working/src/backend")

# The lock pins the CPU build of torch (for laptops). Here we need CUDA, and the wheel has to
# match Kaggle's driver: PyPI's torch 2.15 is a CUDA 13 build (driver >= 580); older drivers
# need the CUDA 12.8 build, which stops at torch 2.11. Only inference runs through torch.
driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.split()
major = int(driver[0].split(".")[0]) if driver else 0
print("NVIDIA driver:", driver[0] if driver else "none found")
assert major, "No GPU found: turn on the accelerator (GPU T4 x2) and restart the session"
if major >= 580:
    run("uv", "pip", "install", "--python", PY, "torch==2.15.0")
else:
    run("uv", "pip", "install", "--python", PY, "torch==2.11.0+cu128",
        "--index-url", "https://download.pytorch.org/whl/cu128",
        "--extra-index-url", "https://pypi.org/simple")

probe = ("import torch; n = torch.cuda.device_count(); "
         "assert n >= 1, 'torch sees no GPU'; "
         "[torch.randn(256, 256, device=f'cuda:{i}').matmul(torch.randn(256, 256, device=f'cuda:{i}')).sum().item() for i in range(n)]; "
         "print(torch.__version__, 'GPUs:', n, 'matmul OK on each')")
run(PY, "-c", probe)
GPUS = int(subprocess.run([PY, "-c", "import torch; print(torch.cuda.device_count())"],
                          capture_output=True, text=True, check=True).stdout)

In [ ]:
import json
gpus = [f"cuda:{i}" for i in range(GPUS)]
env = {
    "DATASET_MODE": "full",
    "DATA_DIR": f"{TMP}/data",
    "FULL_MIN_VIEWS": str(FULL_MIN_VIEWS),
    "CLICKSTREAM_MONTHS": CLICKSTREAM_MONTHS,
    "MAX_CHUNKS_PER_ARTICLE": str(MAX_CHUNKS_PER_ARTICLE),
    "LEIDEN_RESOLUTIONS": json.dumps(LEIDEN_RESOLUTIONS),
    "EMBED_DEVICES": json.dumps(gpus),
    "EMBED_BATCH_SIZE": "256",
    "EMBED_SHARD_ROWS": "100000",
    "DAGSTER_HOME": f"{TMP}/dagster",
    "TOKENIZERS_PARALLELISM": "false",
}
os.environ.update(env)
os.makedirs(env["DATA_DIR"], exist_ok=True)
os.makedirs(env["DAGSTER_HOME"], exist_ok=True)
print(json.dumps(env, indent=1))

In [ ]:
# Everything except the Postgres load (13 assets + their data checks), one step at a time
# so the RAM-heavy steps (UMAP, Leiden, embeddings) never overlap on Kaggle's ~29 GB.
open("/kaggle/working/run.yaml", "w").write("execution:\n  config:\n    multiprocess:\n      max_concurrent: 1\n")
!cd /kaggle/working/src/backend && {VENV}/bin/dagster asset materialize -m rabbithole.pipelines.definitions \
    -c /kaggle/working/run.yaml \
    --select '*map_coords,*rh_scores,*chunk_embeddings,*label_edges,*edges_pruned'

In [ ]:
# Bundle what serving, LTR and evaluation need (article bodies are left behind).
!cd /kaggle/working/src/backend && {VENV}/bin/rabbithole pack --out /kaggle/working/rabbithole_full
!cat /kaggle/working/rabbithole_full/manifest.json
!du -sh /kaggle/working/rabbithole_full
# Free the output folder of everything else, so only the bundle is saved.
!rm -rf /kaggle/working/src /kaggle/working/run.yaml

## Next, on your machine
```bash
pip install kaggle   # once; put your API token in ~/.kaggle/kaggle.json
kaggle kernels output <your-username>/<this-notebook-slug> -p ~/Downloads/rabbithole_full
cd rabbithole
uv --directory backend run rabbithole unpack ~/Downloads/rabbithole_full/rabbithole_full
DATASET_MODE=full make load      # Postgres (streams chunks; ~20–40 min)
```